In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"/UgandaLSMS")

waves = {
    1: ROOT / "Wave_1",
    2: ROOT / "Wave_2",
    3: ROOT / "Wave_3",
    4: ROOT / "Wave_4",
    5: ROOT / "Wave_5",
    7: ROOT / "Wave_7",
    8: ROOT / "Wave_8",
}

In [2]:
rows = []

for wave, folder in waves.items():
    for file in folder.rglob("*.dta"):
        rows.append({
            "wave": wave,
            "file_name": file.name,
            "file_stem": file.stem,
            "folder": file.parent.name,
            "path": str(file)
        })

catalog = pd.DataFrame(rows)

catalog = catalog.sort_values(["wave", "file_stem"]).reset_index(drop=True)

catalog.head(20)

,wave,file_name,file_stem,folder,path
0,1,AGSEC1.dta,AGSEC1,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...
1,1,AGSEC10.dta,AGSEC10,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...
2,1,AGSEC2.dta,AGSEC2,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...
3,1,AGSEC2A.dta,AGSEC2A,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...
4,1,AGSEC2B.dta,AGSEC2B,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...
5,1,AGSEC3A.dta,AGSEC3A,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...
6,1,AGSEC3B.dta,AGSEC3B,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...
7,1,AGSEC4A.dta,AGSEC4A,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...
8,1,AGSEC4B.dta,AGSEC4B,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...
9,1,AGSEC5A.dta,AGSEC5A,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...


In [3]:

def categorize_file(file_stem):
    name = file_stem.lower()

    if name.startswith("gsec"):
        return "household"
    elif name.startswith("agsec"):
        return "agriculture"
    elif name.startswith("csec"):
        return "community"
    elif "geo" in name:
        return "geovars"
    elif "pov" in name or "welfare" in name:
        return "poverty_welfare"
    else:
        return "other"

catalog["category"] = catalog["file_stem"].apply(categorize_file)

catalog

,wave,file_name,file_stem,folder,path,category
0,1,AGSEC1.dta,AGSEC1,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...,agriculture
1,1,AGSEC10.dta,AGSEC10,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...,agriculture
2,1,AGSEC2.dta,AGSEC2,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...,agriculture
3,1,AGSEC2A.dta,AGSEC2A,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...,agriculture
4,1,AGSEC2B.dta,AGSEC2B,Wave_1,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...,agriculture
...,...,...,...,...,...,...
744,8,WSEC4.dta,WSEC4,Wave_8,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...,other
745,8,WSEC5.dta,WSEC5,Wave_8,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...,other
746,8,WSEC6.dta,WSEC6,Wave_8,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...,other
747,8,WSEC7.dta,WSEC7,Wave_8,C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\W...,other


In [4]:
pd.crosstab(catalog["wave"], catalog["category"])

category,agriculture,community,geovars,household,other,poverty_welfare
wave,,,,,,
1,21,40,1,30,2,1
2,16,47,1,29,4,1
3,21,48,1,28,5,0
4,23,50,1,34,6,1
5,29,53,1,31,5,1
7,19,51,1,29,8,1
8,22,48,0,30,8,1


In [8]:
# inspection block

def load_file(row_number):
    row = catalog.loc[row_number]

    print("Wave:", row["wave"])
    print("File:", row["file_name"])
    print("Category:", row["category"])
    print("Path:", row["path"])

    df = pd.read_stata(row["path"], convert_categoricals=False)
    df.columns = df.columns.str.strip()

    print("Shape:", df.shape)

    return df

In [9]:
# use inspection(n)

df = load_file(0)
df.head(10)

Wave: 1
File: AGSEC1.dta
Category: agriculture
Path: C:\Users\Carl\Desktop\CSB_project\UgandaLSMS\Wave_1\AGSEC1.dta
Shape: (2428, 20)


,Year,Hhid,A2aq1,A2bq1,A4aq1,A4bq1,A6aq1,A6aq21,A6aq22,A6bq1,A6cq1,A8,A8q11,A8q12,A10q11,A10q12,A10q13,A10q14,A10q15,A19q16
0,2009,1013000201,2.0,NaN,NaN,NaN,2.0,NaN,,1.0,1.0,2.0,NaN,NaN,1.0,2.0,2.0,NaN,2.0,2.0
1,2009,1013000204,2.0,1.0,NaN,1.0,2.0,NaN,,1.0,1.0,1.0,2.0,NaN,1.0,2.0,1.0,2.0,2.0,2.0
2,2009,1013000210,2.0,2.0,NaN,2.0,2.0,NaN,,1.0,2.0,2.0,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0
3,2009,101300021302,2.0,2.0,NaN,NaN,2.0,NaN,,1.0,2.0,2.0,NaN,NaN,1.0,2.0,2.0,NaN,2.0,2.0
4,2009,1021000108,1.0,2.0,1.0,1.0,2.0,NaN,,2.0,2.0,2.0,NaN,NaN,1.0,2.0,1.0,2.0,1.0,2.0
5,2009,1021000111,2.0,2.0,2.0,2.0,2.0,NaN,,2.0,1.0,2.0,NaN,NaN,1.0,2.0,2.0,NaN,2.0,2.0
6,2009,1021000113,1.0,2.0,1.0,1.0,2.0,NaN,,2.0,1.0,1.0,2.0,NaN,2.0,2.0,2.0,NaN,2.0,2.0
7,2009,1021000408,1.0,2.0,1.0,NaN,2.0,NaN,,2.0,2.0,2.0,NaN,NaN,2.0,2.0,2.0,NaN,2.0,2.0
8,2009,1021000710,1.0,2.0,1.0,1.0,2.0,NaN,,1.0,1.0,2.0,NaN,NaN,2.0,2.0,2.0,NaN,2.0,2.0
9,2009,1021000807,1.0,2.0,1.0,1.0,1.0,1.0,,2.0,2.0,1.0,2.0,NaN,1.0,2.0,1.0,2.0,1.0,2.0


In [10]:
def inspect_file(row_number):
    row = catalog.loc[row_number]

    df, meta = pd.read_stata(
        row["path"],
        convert_categoricals=False,
        iterator=True
    ).read(), pd.read_stata(
        row["path"],
        convert_categoricals=False,
        iterator=True
    )

    info = pd.DataFrame({
        "var_name": df.columns,
        "dtype": [df[c].dtype for c in df.columns],
        "n_missing": [df[c].isna().sum() for c in df.columns],
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    })

    print("Wave:", row["wave"])
    print("File:", row["file_name"])
    print("Shape:", df.shape)

    return info

In [13]:
var_info = inspect_file(10)
var_info.head(15)

Wave: 1
File: AGSEC5B.dta
Shape: (15273, 34)


,var_name,dtype,n_missing,n_unique
0,Year,int16,0,1
1,Hhid,str,0,2348
2,A5bq1,float64,526,23
3,A5bq3,float64,1392,14
4,A5bq4,str,0,155
5,A5bq5,float64,1977,75
6,A5bq6a,float64,2600,282
7,A5bq6b,float64,5040,47
8,A5bq6c,float64,5028,45
9,A5bq6d,float64,5517,101


In [14]:
var_rows = []

for i, row in catalog.iterrows():
    try:
        df = pd.read_stata(row["path"], convert_categoricals=False)
        df.columns = df.columns.str.strip()

        for col in df.columns:
            var_rows.append({
                "wave": row["wave"],
                "file_name": row["file_name"],
                "file_stem": row["file_stem"],
                "category": row["category"],
                "var_name": col,
                "dtype": str(df[col].dtype),
                "n_missing": df[col].isna().sum(),
                "n_unique": df[col].nunique(dropna=True),
                "n_rows": len(df)
            })

        print(f"Done: wave {row['wave']} - {row['file_name']}")

    except Exception as e:
        print(f"Failed: wave {row['wave']} - {row['file_name']}")
        print(e)

var_catalog = pd.DataFrame(var_rows)

var_catalog.to_excel(ROOT / "variable_catalog_all_waves.xlsx", index=False)

Done: wave 1 - AGSEC1.dta
Done: wave 1 - AGSEC10.dta
Done: wave 1 - AGSEC2.dta
Done: wave 1 - AGSEC2A.dta
Done: wave 1 - AGSEC2B.dta
Done: wave 1 - AGSEC3A.dta
Done: wave 1 - AGSEC3B.dta
Done: wave 1 - AGSEC4A.dta
Done: wave 1 - AGSEC4B.dta
Done: wave 1 - AGSEC5A.dta
Done: wave 1 - AGSEC5B.dta
Done: wave 1 - AGSEC6A.dta
Done: wave 1 - AGSEC6B.dta
Done: wave 1 - AGSEC6C.dta
Done: wave 1 - AGSEC7.dta
Done: wave 1 - AGSEC8.dta
Done: wave 1 - AGSEC9A.dta
Done: wave 1 - AGSEC9B.dta
Done: wave 1 - AGSEC9C.dta
Done: wave 1 - AGSEC9D.dta
Done: wave 1 - AGSEC9E.dta
Done: wave 1 - CSEC1.dta
Done: wave 1 - CSEC2A.dta
Done: wave 1 - CSEC2B.dta
Done: wave 1 - CSEC2C.dta
Done: wave 1 - CSEC3.dta
Done: wave 1 - CSEC3A.dta
Done: wave 1 - CSEC3B.dta
Done: wave 1 - CSEC3C.dta
Done: wave 1 - CSEC3D.dta
Done: wave 1 - CSEC3E.dta
Done: wave 1 - CSEC3F.dta
Done: wave 1 - CSEC3G.dta
Done: wave 1 - CSEC3H.dta
Done: wave 1 - CSEC3I.dta
Done: wave 1 - CSEC3J.dta
Done: wave 1 - CSEC3K.dta
Done: wave 1 - CSEC4.dt

In [15]:
def inspect(row_number, n=5):
    row = catalog.loc[row_number]

    df = pd.read_stata(row["path"], convert_categoricals=False)
    df.columns = df.columns.str.strip()

    print("Row number:", row_number)
    print("Wave:", row["wave"])
    print("File:", row["file_name"])
    print("Category:", row["category"])
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    display(df.head(n))

    return df

In [33]:
df = inspect(2)

Row number: 2
Wave: 1
File: AGSEC2.dta
Category: agriculture
Shape: (2428, 20)

Columns:
['Year', 'Hhid', 'A2aq1', 'A2bq1', 'A4aq1', 'A4bq1', 'A6aq1', 'A6aq21', 'A6aq22', 'A6bq1', 'A6cq1', 'A8', 'A8q11', 'A8q12', 'A10q11', 'A10q12', 'A10q13', 'A10q14', 'A10q15', 'A19q16']


,Year,Hhid,A2aq1,A2bq1,A4aq1,A4bq1,A6aq1,A6aq21,A6aq22,A6bq1,A6cq1,A8,A8q11,A8q12,A10q11,A10q12,A10q13,A10q14,A10q15,A19q16
0,2009,1013000201,2.0,NaN,NaN,NaN,2.0,NaN,,1.0,1.0,2.0,NaN,NaN,1.0,2.0,2.0,NaN,2.0,2.0
1,2009,1013000204,2.0,1.0,NaN,1.0,2.0,NaN,,1.0,1.0,1.0,2.0,NaN,1.0,2.0,1.0,2.0,2.0,2.0
2,2009,1013000210,2.0,2.0,NaN,2.0,2.0,NaN,,1.0,2.0,2.0,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0
3,2009,101300021302,2.0,2.0,NaN,NaN,2.0,NaN,,1.0,2.0,2.0,NaN,NaN,1.0,2.0,2.0,NaN,2.0,2.0
4,2009,1021000108,1.0,2.0,1.0,1.0,2.0,NaN,,2.0,2.0,2.0,NaN,NaN,1.0,2.0,1.0,2.0,1.0,2.0


In [17]:
def check_ids(df):
    possible_ids = [
        "HHID", "hhid", "Hhid",
        "PID", "pid",
        "COMCOD", "comcod",
        "HHID_old", "hhid_old",
        "parcelID", "plotID", "cropID"
    ]

    rows = []

    for col in df.columns:
        if col in possible_ids or "hh" in col.lower() or "id" in col.lower() or "com" in col.lower():
            rows.append({
                "column": col,
                "dtype": str(df[col].dtype),
                "n_missing": df[col].isna().sum(),
                "n_unique": df[col].nunique(dropna=True),
                "example_values": df[col].dropna().astype(str).head(3).tolist()
            })

    return pd.DataFrame(rows)



In [18]:
check_ids(df)

,column,dtype,n_missing,n_unique,example_values
0,Hhid,str,0,2428,"[1013000201, 1013000204, 1013000210]"


In [25]:
def check_level(df, keys):
    keys = [k.upper() for k in keys]
    keys = [k for k in keys if k in df.columns]

    if not keys:
        print("No valid keys found.")
        return None

    n_rows = len(df)
    n_unique = df[keys].drop_duplicates().shape[0]
    n_duplicates = n_rows - n_unique

    print("Keys:", keys)
    print("Rows:", n_rows)
    print("Unique key combinations:", n_unique)
    print("Duplicate key combinations:", n_duplicates)

    if n_duplicates == 0:
        print("Level: unique at this level")
    else:
        print("Level: multiple rows per key")

    return df[df.duplicated(keys, keep=False)].sort_values(keys).head(20)

In [27]:
check_level(df, ["HHID"])

check_level(df, ["HHID", "PID", "pid", "Pid"])

check_level(df, ["COMCOD"])

Keys: ['HHID']
Rows: 2428
Unique key combinations: 2428
Duplicate key combinations: 0
Level: unique at this level
Keys: ['HHID']
Rows: 2428
Unique key combinations: 2428
Duplicate key combinations: 0
Level: unique at this level
No valid keys found.


In [23]:
def load_clean(row_number):
    row = catalog.loc[row_number]

    df = pd.read_stata(row["path"], convert_categoricals=False)

    # Standardize all column names
    df.columns = df.columns.str.strip().str.upper()

    df["WAVE"] = row["wave"]

    # Clean HHID if it exists
    if "HHID" in df.columns:
        df["HHID"] = (
            df["HHID"]
            .astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
        )

    return df

In [24]:
df = load_clean(0)

check_level(df, ["HHID"])
check_level(df, ["HHID", "PID"])
check_level(df, ["COMCOD"])

Keys: ['HHID']
Rows: 2428
Unique key combinations: 2428
Duplicate key combinations: 0
Level: unique at this level
Keys: ['HHID']
Rows: 2428
Unique key combinations: 2428
Duplicate key combinations: 0
Level: unique at this level
No valid keys found.


In [34]:
ag_crosswalk = {
    "AGSEC1": {
        1: ["AGSEC1"],
        3: ["AGSEC1"],
        4: ["AGSEC1"],
        5: ["AGSEC1"],
        7: ["AGSEC1"],
        8: ["AGSEC1"],
    },

    "AGSEC2": {
        1: ["AGSEC2"],
        2: ["AGSEC2"],
    },

    "AGSEC2A": {
        1: ["AGSEC2A"],
        2: ["AGSEC2A"],
        3: ["AGSEC2A"],
        4: ["AGSEC2A"],
        5: ["AGSEC2A", "AGSEC2AB_1"],
        7: ["AGSEC2A"],
        8: ["AGSEC2A"],
    },

    "AGSEC2B": {
        1: ["AGSEC2B"],
        2: ["AGSEC2B"],
        3: ["AGSEC2B"],
        4: ["AGSEC2B"],
        5: ["AGSEC2B"],
        7: ["AGSEC2B"],
        8: ["AGSEC2B"],
    },

    "AGSEC3A": {
        1: ["AGSEC3A"],
        2: ["AGSEC3A"],
        3: ["AGSEC3A"],
        4: ["AGSEC3A"],
        5: ["AGSEC3A", "AGSEC3A_1"],
        7: ["AGSEC3A"],
        8: ["AGSEC3A", "AGSEC3A_1"],
    },

    "AGSEC3B": {
        1: ["AGSEC3B"],
        2: ["AGSEC3B"],
        3: ["AGSEC3B"],
        4: ["AGSEC3B"],
        5: ["AGSEC3B", "AGSEC3B_1"],
        7: ["AGSEC3B"],
        8: ["AGSEC3B", "AGSEC3B_1"],
    },

    "AGSEC4A": {
        1: ["AGSEC4A"],
        2: ["AGSEC4A"],
        3: ["AGSEC4A"],
        4: ["AGSEC4A"],
        5: ["AGSEC4A", "AGSEC4A_1"],
        7: ["AGSEC4A"],
        8: ["AGSEC4A"],
    },

    "AGSEC4B": {
        1: ["AGSEC4B"],
        2: ["AGSEC4B"],
        3: ["AGSEC4B"],
        4: ["AGSEC4B"],
        5: ["AGSEC4B", "AGSEC4B_1"],
        7: ["AGSEC4B"],
        8: ["AGSEC4B"],
    },

    "AGSEC5A": {
        1: ["AGSEC5A"],
        2: ["AGSEC5A"],
        3: ["AGSEC5A"],
        4: ["AGSEC5A"],
        5: ["AGSEC5A"],
        7: ["AGSEC5A"],
        8: ["AGSEC5A"],
    },

    "AGSEC5B": {
        1: ["AGSEC5B"],
        2: ["AGSEC5B"],
        3: ["AGSEC5B"],
        4: ["AGSEC5B"],
        5: ["AGSEC5B"],
        7: ["AGSEC5B"],
        8: ["AGSEC5B"],
    },

    "AGSEC6A": {
        1: ["AGSEC6A"],
        2: ["AGSEC6A"],
        3: ["AGSEC6A"],
        4: ["AGSEC6A", "AGSEC6A_1"],
        5: ["AGSEC6A", "AGSEC6A_1"],
        7: ["AGSEC6A"],
        8: ["AGSEC6A"],
    },

    "AGSEC6B": {
        1: ["AGSEC6B"],
        2: ["AGSEC6B"],
        3: ["AGSEC6B"],
        4: ["AGSEC6B", "AGSEC6B_1"],
        5: ["AGSEC6B", "AGSEC6B_1"],
        7: ["AGSEC6B"],
        8: ["AGSEC6B"],
    },

    "AGSEC6C": {
        1: ["AGSEC6C"],
        2: ["AGSEC6C"],
        3: ["AGSEC6C"],
        4: ["AGSEC6C", "AGSEC6C_1"],
        5: ["AGSEC6C", "AGSEC6C_1"],
        7: ["AGSEC6C"],
        8: ["AGSEC6C"],
    },

    "AGSEC7": {
        1: ["AGSEC7"],
        2: ["AGSEC7"],
        4: ["AGSEC7"],
        8: ["AGSEC7"],
    },

    "AGSEC7A": {
        3: ["AGSEC7A"],
        5: ["AGSEC7A"],
    },

    "AGSEC7B": {
        3: ["AGSEC7B"],
        5: ["AGSEC7B"],
    },

    "AGSEC8": {
        1: ["AGSEC8"],
        2: ["AGSEC8"],
    },

    "AGSEC8A": {
        3: ["AGSEC8A"],
        4: ["AGSEC8A"],
        5: ["AGSEC8A"],
        7: ["AGSEC8A"],
        8: ["AGSEC8A"],
    },

    "AGSEC8B": {
        3: ["AGSEC8B"],
        4: ["AGSEC8B"],
        5: ["AGSEC8B"],
        7: ["AGSEC8B"],
        8: ["AGSEC8B"],
    },

    "AGSEC8C": {
        3: ["AGSEC8C"],
        4: ["AGSEC8C"],
        5: ["AGSEC8C"],
        7: ["AGSEC8C"],
        8: ["AGSEC8C"],
    },

    "AGSEC8D": {
        3: ["AGSEC8D"],
    },

    "AGSEC8E": {
        3: ["AGSEC8E"],
    },

    "AGSEC9": {
        2: ["AGSEC9"],
        3: ["AGSEC9"],
        4: ["AGSEC9"],
    },

    "AGSEC9A": {
        1: ["AGSEC9A"],
        4: ["AGSEC9A"],
        5: ["AGSEC9A"],
        7: ["AGSEC9A"],
        8: ["AGSEC9A"],
    },

    "AGSEC9B": {
        1: ["AGSEC9B"],
        5: ["AGSEC9B"],
        7: ["AGSEC9B"],
        8: ["AGSEC9B"],
    },

    "AGSEC9C": {
        1: ["AGSEC9C"],
    },

    "AGSEC9D": {
        1: ["AGSEC9D"],
    },

    "AGSEC9E": {
        1: ["AGSEC9E"],
    },

    "AGSEC10": {
        1: ["AGSEC10"],
        2: ["AGSEC10"],
        3: ["AGSEC10"],
        4: ["AGSEC10"],
        5: ["AGSEC10"],
        7: ["AGSEC10"],
        8: ["AGSEC10"],
    },

    "AGSEC11": {
        4: ["AGSEC11"],
        5: ["AGSEC11"],
        7: ["AGSEC11"],
        8: ["AGSEC11"],
    },
}

In [28]:
#find matching files
def find_file(wave, file_stem):
    matches = catalog[
        (catalog["wave"] == wave) &
        (catalog["file_stem"].str.upper() == file_stem.upper())
    ]

    if len(matches) == 0:
        print(f"Missing: wave {wave}, {file_stem}")
        return None

    if len(matches) > 1:
        print(f"Multiple matches: wave {wave}, {file_stem}")
        display(matches)

    return matches.iloc[0]

In [35]:
def load_ag_section(canonical_section):
    section_map = ag_crosswalk[canonical_section]

    parts = []

    for wave, file_stems in section_map.items():
        for file_stem in file_stems:

            row = find_file(wave, file_stem)

            if row is None:
                continue

            df = pd.read_stata(row["path"], convert_categoricals=False)

            # standardize column names
            df.columns = df.columns.str.strip().str.upper()

            # add metadata
            df["WAVE"] = wave
            df["SOURCE_FILE"] = row["file_stem"]
            df["CANONICAL_SECTION"] = canonical_section

            # clean HHID if present
            if "HHID" in df.columns:
                df["HHID"] = (
                    df["HHID"]
                    .astype(str)
                    .str.strip()
                    .str.replace(r"\.0$", "", regex=True)
                )

            parts.append(df)

            print(f"Loaded wave {wave}: {file_stem}, shape={df.shape}")

    if not parts:
        print(f"No files loaded for {canonical_section}")
        return None

    out = pd.concat(parts, ignore_index=True, sort=False)

    print()
    print(f"Final stacked {canonical_section}: {out.shape}")

    return out

In [36]:
# use wavecross section load

agsec1 = load_ag_section("AGSEC1")
agsec2a = load_ag_section("AGSEC2A")
agsec8 = load_ag_section("AGSEC8")



Loaded wave 1: AGSEC1, shape=(2428, 23)
Loaded wave 3: AGSEC1, shape=(2277, 25)
Loaded wave 4: AGSEC1, shape=(2495, 29)
Loaded wave 5: AGSEC1, shape=(2694, 39)
Loaded wave 7: AGSEC1, shape=(3242, 18)
Loaded wave 8: AGSEC1, shape=(2586, 15)

Final stacked AGSEC1: (15722, 90)
Loaded wave 1: AGSEC2A, shape=(4305, 37)
Loaded wave 2: AGSEC2A, shape=(3457, 38)
Loaded wave 3: AGSEC2A, shape=(3763, 39)
Loaded wave 4: AGSEC2A, shape=(4142, 42)
Loaded wave 5: AGSEC2A, shape=(4200, 41)
Loaded wave 5: AGSEC2AB_1, shape=(2694, 5)
Loaded wave 7: AGSEC2A, shape=(4368, 50)
Loaded wave 8: AGSEC2A, shape=(4087, 49)

Final stacked AGSEC2A: (31016, 111)
Loaded wave 1: AGSEC8, shape=(5203, 16)
Loaded wave 2: AGSEC8, shape=(7350, 15)

Final stacked AGSEC8: (12553, 16)


In [39]:
agsec1.shape

agsec1[["WAVE", "SOURCE_FILE", "CANONICAL_SECTION"]].value_counts().sort_index()

pd.DataFrame({
    "column": agsec1.columns,
    "non_missing": agsec1.notna().sum().values,
    "dtype": [str(agsec1[c].dtype) for c in agsec1.columns]
})

check_ids(agsec1)

check_level(agsec1, ["HHID"])
check_level(agsec1, ["HHID", "PLOTID"])
check_level(agsec1, ["HHID", "PARCELID"])



Keys: ['HHID']
Rows: 15722
Unique key combinations: 13656
Duplicate key combinations: 2066
Level: multiple rows per key
Keys: ['HHID']
Rows: 15722
Unique key combinations: 13656
Duplicate key combinations: 2066
Level: multiple rows per key
Keys: ['HHID']
Rows: 15722
Unique key combinations: 13656
Duplicate key combinations: 2066
Level: multiple rows per key


,YEAR,HHID,A2AQ1,A2BQ1,A4AQ1,A4BQ1,A6AQ1,A6AQ21,A6AQ22,A6BQ1,...,HH_CRP2,LVSTCK,HH_ANM,HH_PLTY,T0_HHID,BATCH,HWGT_W7,HHSIZE,HWGT_WC,CULCRP
2,2009.0,1013000210,2.0,2.0,NaN,2.0,2.0,NaN,,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2428,NaN,1013000210,2.0,NaN,NaN,NaN,2.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2009.0,1021000710,1.0,2.0,1.0,1.0,2.0,NaN,,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2429,NaN,1021000710,1.0,NaN,NaN,NaN,2.0,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,2009.0,102100080803,1.0,1.0,1.0,1.0,1.0,1.0,,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4396,NaN,102100080803,1.0,NaN,NaN,NaN,1.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,2009.0,1021001109,1.0,2.0,1.0,1.0,1.0,1.0,,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2430,NaN,1021001109,1.0,NaN,NaN,NaN,2.0,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,2009.0,1021002210,2.0,2.0,2.0,NaN,2.0,NaN,,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2433,NaN,1021002210,1.0,NaN,NaN,NaN,2.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
#save successful stack

OUT = ROOT / "intermediate_ag_sections"
OUT.mkdir(exist_ok=True)

agsec1.head(5000).to_excel(OUT / "AGSEC1_stacked_preview.xlsx", index=False)
